<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Lecture 6: Sentence Meaning

**University of Maryland, College Park**  
**Fall 2025**  
**Instructor**: Armin Mehrabian  

## 🎯 Learning Objectives

By the end of this lecture, you will be able to:
1. Understand meaning representation and meaning representation languages
2. Distinguish between procedural and model-theoretic semantics
3. Design and apply meaning representation languages for sentences
4. Understand semantic role labeling and predicate-argument structure
5. Work with Abstract Meaning Representation (AMR)
6. Apply compositional semantic analysis using lambda notation
7. Understand neural sequence models for sentence representation
8. Implement RNNs, LSTMs, and BERT for semantic tasks

## 📚 Setup and Imports

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP libraries
import nltk
import spacy

# Deep learning
import torch
import torch.nn as nn

# Download required NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except:
    print("Please install spaCy model: python -m spacy download en_core_web_sm")

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Meaning Representation

## What is Meaning Representation?

A meaning of a linguistic structure can be captured in a formal structure called **meaning representation**.

The framework that specifies the syntax and semantic of this representation is called **meaning representation languages**.

Need for a representation that can bridge the gap from linguistic inputs to the non-linguistic knowledge of the world, to perform tasks that involve the meaning of linguistic inputs.

## Tasks Requiring Semantic Processing

Following language tasks require some form of semantic processing of natural language:

- Answering essay questions on an exam
- Deciding what to order at a restaurant by reading a menu
- Learning to use a new piece of software by reading the manual
- Realizing that you've been insulted
- Following recipes

**Key insight**: Linguistic elements → Knowledge of the world

# 2. Programming Analogy

## Functions and Arguments

In programming, we often talk about a function and its arguments:

- `append(item, list)` → `list`
  - e.g. `append(1, [3,2])` → `[3,2,1]`
  - Arguments are `(1, [3,2])`, function is `append`, result is `[3,2,1]`

- Arguments can have types to which they are restricted
  - `append(1, 6)` results in an error, since 6 is not a list

- Note that Boolean functions can be interpreted as relations
  - `append(A,B)` is true if A is an (item, list) pair and B is the list with that item appended to it
  - append is a relation and the pair `<(1,[3,2]), [3,2,1]>` is a member of that relation

# 3. Meaning, Arguments, and Predicates

When we represent meaning, we similarly distinguish predicates and arguments

**Example**: *Philip likes Cristina's that is a noisy Italian restaurant in Silver Spring, Maryland.*

- **Restaurant** (Cristinas)
- **Likes** (Philip, Cristinas)
- **Noisy** (Cristinas)
- **Location** (Philip, Cristinas)

## Procedural Semantics

### Meaning as a Procedure

**Meaning = Procedure for making something true**

`[[ Location(Philip, Cristinas) ]] =`
- Philip gets in car
- Philip looks up Cristinas on Google Maps
- Philip starts car
- Philip drives following directions to Cristinas

### Procedural Semantics: Verification

**Meaning = Procedure for verifying that something is true**

`[[ Location(Philip, Cristinas) ]] =`
- Look up Philip's iPhone GPS location x
- Use Google Maps to get Cristinas GPS location y
- Calculate distance between x and y
- If distance is less than 10 yards, return TRUE, else return FALSE

## SHRDLU Example (Winograd 1972)

Uses inference to figure out what world has to be true to make something true, then make those things true.

Good illustration of the study of natural language understanding using "microworlds".

**Dialogue Example**:
```
Person: Pick up a big red block.
Computer: OK.

Person: Grasp the pyramid.
Computer: I DON'T UNDERSTAND WHICH PYRAMID YOU MEAN.

Person (changing their mind): Find a block which is taller than the 
one you are holding and put it into the box.
Computer: BY "IT", I ASSUME YOU MEAN THE BLOCK WHICH IS 
TALLER THAN THE ONE I AM HOLDING.
Computer: OK.
```

# 4. Model-theoretic Semantics

## Meaning as a Homomorphism

- Assume there is some model that is an abstraction of the world
- The meaning of a sentence expressed in a **meaning representation language** is a **truth value (true/false)** determined by mapping from the meaning representation to the model.

### Example: "A brown dog in Central Park chases a squirrel"

#### Model (M)
**Domain**: D = {a, e, h}
- Dog: a
- Squirrel: e
- Central_Park: h

**Properties**
- Brown = {a} (brown dog)

**Relations**
- Chases m: Chases = {<a,e>} (brown dog chases a squirrel)

**Modifiers of verb**
- Location: Loc = {<m,h>} (chases in Central_park)

#### Meaning Representation (R)
```
^ chase(m)
^ subject (m,a)
^ dog (d)
^ brown (d)
^ loc (m,h)
^ squirrel (e)
^ object (m,e)
^ central_park (h)
```

## Verifying Truth

**Question**: Is the sentence true?

**Answer**: Is there a mapping from expression R to the model M that preserves the relationships?

```
       m ─────────────────→ M₃
      / \                  / \
  subj   obj          CHASER  CHASEE
   /       \            /       \
  a         e          A         E
```

# 5. Current Semantics Work in Computational Linguistics

## Three Main Areas

1. **Designing meaning representation languages**

2. **Getting from text to meaning representations (semantic analysis)**
   - Traditionally via syntactic analysis
   - More recently using deep learning to go straight from language to meaning representations

3. **Doing things with semantic representations, especially inference**

# 6. Designing Meaning Representation Languages

## Semantic Roles

Key emphasis on **semantic roles**

**Predicate-argument structure** – typically events and participants in that event

### Example
```
Leo the lion ate Remy on Tuesday
     arg0      rel arg1  argM-TMP

Remy was eaten by Leo the lion on Tuesday
arg1           arg0              argM-TMP
```

**Components**:
- **Rel**: verb
- **Argument of verb**: arg0, arg1, ...
- **Modifiers of verb**: manner (MNR), location (LOC), temporal (TMP)

## PropBank and FrameNet

### PropBank (Proposition Bank)
A corpus annotated with semantic roles for predicates using a linguistically motivated design of the roles:
- **arg0**: agent, doer
- **arg1**: theme, done-to
- **arg2**: instrument (e.g., with a fork)
- ...

### FrameNet
A lexical database/corpus annotation project that captures conceptual structure for an event with roles and associated roles:
- **Frame**: Commerce-Goods-Transfer  
  - Lexical units: buy, sell, purchase
- **Core elements**: Buyer, Seller, Goods
- **Non-Core elements**: Place, Purpose

# 7. Abstract Meaning Representation (AMR)

## Overview

AMR combines several key features:
- PropBank frame sets as concepts
- Semantic roles from PropBank
- Named entities with types
- Coreference
- Unified Verb Index maps with FrameNet, VerbNet

### Example: "China is expanding its military power to attempt to join the ranks of the superpowers"

```
                expand-01
                   |
            ┌──────┴──────┐
          ARG0          ARG1
            |             |
         country        power
            |             |
          name          mod
            |             |
          name        military
            |
           op1
            |
          China

         purpose: attempt-01
                      |
                   ARG1: join-01
                            |
                         ARG1: rank
                                |
                              mod: superpower
```

## Constraints on Arguments

Similar idea to argument types in programming languages

**Analogy**: in `append(x, L)`, L must be a list

### Traditional approach: Boolean restrictions
- `devour(x, y)`
  - X should be an animal, y must be edible
- **Problem**: too strict
  - "Jay devoured The Hobbit"
  - "This class is devouring my spirit"

## Information-theoretic Approach (Resnik 1993, 1996)

- Consider C to be a semantic category, e.g. in WordNet
- Estimate Pr(C|V=devour); compare to Pr(C)
- Compute KL-divergence between Pr(C|V=v) and Pr(C): how well does Pr(C) estimate Pr(C|V=v)?
- This "selectional association" between verb v and argument category C characterizes degree of constraint
- High when C is a good semantic fit for that argument of v.

# 8. Semantic Analysis

## Semantic Role Labeling (SRL)

SRL is one crucial piece of the puzzle

### Can be interpreted as a classification problem

**Example**: [The NY Times] issued [a special solution] [yesterday]

- Is [The NY Times] an argument? (Boolean label)
- What is its role? (arg0, arg1, etc.) (Choice among multiple labels)

### Can also be treated as a sequence labeling problem

Directly analogous to POS tagging, with begins/inside/outside (B,I,O) labels

```
[The NY Times] issued [a special solution] [yesterday]
 B-arg0 I-arg0 I-arg0   O   B-arg1 I-arg1 I-arg1  B-argM-TMP
```

In [ ]:
# Demo: Semantic Role Labeling with spaCy
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Example sentence
sentence = "The NY Times issued a special solution yesterday"
doc = nlp(sentence)

print("Dependency-based role analysis:")
print(f"{'Token':<15} {'Dependency':<10} {'Head':<15} {'Role Interpretation'}")
print("="*70)

for token in doc:
    role = ""
    if token.dep_ == "nsubj":
        role = "ARG0 (agent)"
    elif token.dep_ == "dobj":
        role = "ARG1 (theme)"
    elif token.dep_ == "npadvmod" and token.pos_ == "NOUN":
        role = "ARGM-TMP (temporal)"
    
    print(f"{token.text:<15} {token.dep_:<10} {token.head.text:<15} {role}")

# 9. Compositional Semantic Analysis

## Lambda Notation

A function that takes argument x and yields P(x):

`λx · P(x)` ⇒ Greek symbol λ, one or more variable x, and First Order Logic formula

## Lambda-reduction

Ability to apply them to logical terms to yield new FOL; λ-reduction

```
λx · P(x)(A)
    ⇓
   P(A)
```

### Example: Using λ-expression as body of another
```
λx.λy.Near(x,y)
λx.λy.Near(x,y)(Bacaro)
λy.Near(y)(Bacaro)(Centro)
Near(Bacaro,Centro)
```

## Programming Language Analogy

Just like in programming languages:
```python
map(lambda x: x*2, [1,2,3]) → [2,4,6]
```

In [ ]:
# Demo: Lambda functions in Python (analogy to semantic λ-calculus)

# Simple lambda function
double = lambda x: x * 2
print("Lambda x: x*2 applied to [1,2,3]:")
print(list(map(double, [1,2,3])))

# Curried lambda (multi-argument)
near = lambda x: lambda y: f"Near({x},{y})"
print("\nCurried lambda Near:")
print(near("Bacaro")("Centro"))

# Semantic composition example
def compose_meaning(verb, subject, obj):
    """Simplified compositional semantics"""
    return lambda v, s, o: f"{v}({s}, {o})"

give_meaning = compose_meaning("give", "John", "book")
print("\nCompositional semantics:")
print(give_meaning("GIVE", "John", "book"))

# 10. Doing Things with Representations

## Inference with Knowledge Graphs

Inference! Typically against a knowledge-base or knowledge graph

### Example: Question-answering

```
CanPatent(X,Y) ← Inventor(X,Y) ∧ Person(X)
CanPatent(X,Y) ← (Organization(X) ∧ ∃Y (Person(Y) ∧ WorksFor(Y,X))
```

Wide range of logical formalisms supporting inference

# 11. Neural Sequence Models and Sentence Representation

## Non-sequential Models

### Problem:
- Keep adding X₃, X₄, ...?
- Sentences grow unboundedly
- Missing generalizations

```
x → y       y = f(x)

x₁ \
x₂  → y     y = f(x₁, x₂)
```

# 12. Training Recurrent Neural Networks

## Back-propagation in Time

- "Unrolls" to create a very deep network
- Input: typically symbols x are mapped immediately to pre-trained embeddings e

```
                x₂      x₃      x₄      x₅
y* =             a     hole      in     the    .....

Cross-entropy   -log    -log    -log    -log
loss           Pr₁(x₂) Pr₂(x₃) Pr₃(x₄) Pr₄(x₅)

softmax          o       o       o       o      o

                h₁  →   h₂  →   h₃  →   h₄  →  h₅  →

e                □       □       □       □      □   .....
↑
x               In       a     hole      in     the .....
               x₁      x₂      x₃      x₄      x₅  .....
```

**Distributed representation of meaning for the entire sentence up to this point**

## Part-of-Speech Tagging with RNNs

```
y* =            NNP    MD     VB     DT    .....
                y₁     y₂     y₃     y₄

Cross-entropy loss

softmax          o      o      o      o

"RNN layer"    ┌─────────────────────┐
               │ h₁ → h₂ → h₃ → h₄   │ .....
               └─────────────────────┘

                □      □      □      □      □   .....

               x₁     x₂     x₃     x₄     x₅
              Janet  will   back   the    bill  .....
```

## Sequence Classification with RNNs

```
                                    y*
                                    y     Loss function
                                   ⌄ ⌄
                                  ╱   ╲   Feed-forward network
                                 ╱     ╲
                                ├───────┤ hₙ
                                    ↑
        ┌───────────────────────────┤
        │                           │
        │                           │
       X₁   X₂   .....             Xₙ
```

Note: back-propagation from y all the way back through the network **requires** the ability to train deep networks

# 13. Bi-directional RNNs

## Forward and Backward Representations

- Allows hidden representations of a token to take into account both preceding and following context
- Assumes entire string is available at once

```
                y₁      y₂              yₙ₋₁     yₙ
                ↑       ↑                ↑       ↑
        ┌───────┼───┬───┼───┐   ┌───────┼───┬───┼───┐
Backward│   ←  h₁'  │← h₂'  │...│  ← hₙ₋₁'│← hₙ'  │  Bidirectional
        │           │       │   │         │       │  RNN layer
Forward │  h₁  →   │ h₂ →  │...│  hₙ₋₁ → │ hₙ →  │
        └───────┼───┴───┼───┘   └───────┼───┴───┼───┘
                ↑       ↑                ↑       ↑
               x₁      x₂              xₙ₋₁     xₙ
```

## Stacked RNNs

```
                .....
                ↑  ↑  ↑

        h₁' → h₂' → h₃' → ..... →
         ↑     ↑     ↑

        h₁  → h₂  → h₃  → ..... →
         ↑     ↑     ↑

        x₁    x₂    x₃   .....
```

Each of these layers could also be a **bi-directional RNN**

In [ ]:
# Demo: Simple RNN for sequence classification
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [batch_size, seq_len]
        embedded = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        output, hidden = self.rnn(embedded)  # output: [batch_size, seq_len, hidden_dim]
        # Use last hidden state for classification
        prediction = self.fc(hidden.squeeze(0))  # [batch_size, output_dim]
        return prediction

# Example usage
vocab_size = 1000
embedding_dim = 100
hidden_dim = 128
output_dim = 2  # binary classification

model = SimpleRNN(vocab_size, embedding_dim, hidden_dim, output_dim)
print(model)

# Test with dummy data
dummy_input = torch.randint(0, vocab_size, (4, 10))  # batch_size=4, seq_len=10
output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

# 14. Autoregressive Generation of Sequences

```
                      next word sampled
                      from softmax
                         x₂   x₃
        softmax        ┌────────┐
                       │  RNN   │
        embedding      │        │
                       └────────┘
        input word     <s> x₁  x₂  x₃  ...
```

## Generating From Context

```
┌─────────┐                        ┌────────────────┐
│ Encoder │ ──→ C ──→              │ RNN generation │
└─────────┘                        │                │
                                   │      x₂  x₃    │
                                   │       ↑   ↑    │
                                   │  <s> x₁ x₂ x₃ ...│
                                   └────────────────┘
                                        Decoder
```

- C can represent an encoding of prior context using an RNN
- C can also represent encoding of anything else – e.g. an image, so that generation (the decoder) produces a caption

## Encoder-Decoders for Machine Translation

### Encoder
```
S₀ᵉ → [  ] → S₁ᵉ → [  ] → ... → S₅ᵉ = C (vector representation of entire input)
        ↑           ↑             ↑
       X₁          X₂           X₅
      the         cat   in the  house
```

### Decoder
```
       y* = el     y* = gato    y* = casa
           ↑            ↑             ↑
S₀ˡ=C → [  ] → ... → [  ] → ... → [  ] →
           ↑            ↑             ↑
          X₁           X₂            X₅
          the         cat          house
```

In [ ]:
# Demo: Simple Encoder-Decoder architecture
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return hidden  # Context vector C

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, context):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, context)
        prediction = self.fc(output)
        return prediction, hidden

# Create encoder-decoder
encoder = Encoder(1000, 100, 128)
decoder = Decoder(1000, 100, 128)

# Example usage
source = torch.randint(0, 1000, (2, 5))  # batch=2, seq_len=5
context = encoder(source)
print(f"Context shape: {context.shape}")

target_input = torch.randint(0, 1000, (2, 3))  # batch=2, seq_len=3
output, _ = decoder(target_input, context)
print(f"Decoder output shape: {output.shape}")

# 15. Long Short-term Memory (LSTM)

## The Problem

A problem for the original RNN: **earlier words have less influence than later words**

It is therefore more difficult to capture longer-distance dependencies

```
                X₂      X₃      ........   Xₙ
                ↑       ↑                  ↑       ↑
        H₁  →  h2  →   h3  → .... ... Hn-1 → hn
        ↑       ↑       ↑                  ↑       ↑
       X₁      X₂      X₃   ........  Xₙ₋₁     Xₙ

        Less influential              More influential
        in predicting Xₙ              in predicting Xₙ
```

## Long-Distance Dependency Example

```
The flights [ the airline was cancelling after a serious storm knocked out power ] were full
    PL                                                                                PL
    └────────────────────── long-distance dependency ─────────────────────────────────┘
```

## LSTM Solution

LSTM: uses **"gates"** to control propagation of information

Same architecture as RNN, just different internal architecture for the units!

```
       ......         Cₜ₋₁                Cₜ              ......
                    ┌────┐  LSTM unit   ┌────┐
                    │    │    ╱╱╱       │    │
                    │hₜ₋₁│   ╱╱╱╱       │ hₜ │
                    └────┘  ╱           └────┘
                            ↑                   ↑
                           Xₜ                 Xₜ₊₁
```

## LSTM Internal Structure

```
                Cₜ₋₁          LSTM unit          Cₜ
              ┌────┐                           ┌────┐
              │    │                           │    │
     ......   │hₜ₋₁│                           │ hₜ │    ......
              └────┘                           └────┘
                             ↑
                            Xₜ

        ┌─────────────────────────────────────────────┐
        │                                             │
  xₜ ───┼─→ (oₜ) Output Gate                         │
        │                                             │
        │   (iₜ) Input Gate                          │ hₜ
        │      ↓        ⊗        Cell                │  →
        │     ╱─╲  →   (Cₜ)  →  ╱─╲  →  ⊗           │
        │                        ⊙                   │
        │   (fₜ) Forget Gate                         │
        └─────────────────────────────────────────────┘
```

**Key components**:
- **Forget gate** (fₜ): What to forget from cell state
- **Input gate** (iₜ): What new information to store
- **Output gate** (oₜ): What to output from cell state
- **Cell state** (Cₜ): Long-term memory

In [ ]:
# Demo: LSTM vs Simple RNN
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        # Use last hidden state
        prediction = self.fc(hidden.squeeze(0))
        return prediction

# Compare architectures
rnn_model = SimpleRNN(1000, 100, 128, 2)
lstm_model = LSTMClassifier(1000, 100, 128, 2)

print("RNN Parameters:", sum(p.numel() for p in rnn_model.parameters()))
print("LSTM Parameters:", sum(p.numel() for p in lstm_model.parameters()))
print("\nLSTM has more parameters due to gates (forget, input, output)")

# 16. Sequential Neural Models and Syntax

## Recall: Finite State Automata and HMMs

- Finite state automata:
  - Capture observable structure but do not represent latent/abstract categories
  - Do not capture important long-range linguistic dependencies

- HMMs learn latent structure:
  ```
  q₁ → q₂ → q₃    Latent word categories
  ↓    ↓    ↓
  little brown dogs
  ```

## Advantages of Neural Models

With fewer parameters than n-gram models, they are more effective at capturing generalizations:

- **N-gram model**: O(Vⁿ) parameters over a vocabulary of size V
- **HMM with S states**: O(S²) + O(V*S) parameters, where typically S << V

But HMMs are still fundamentally finite-state and therefore fail to model long-distance dependencies

## RNNs and LSTMs: Not Finite-State

In contrast, sequential neural models like RNNs and LSTMs are **not strictly finite-state**:

- **Infinite number of states** can be encoded by continuous-valued hidden state vectors
- Current state can depend on **arbitrarily distant** previous information
- LSTMs further facilitate **"remembering"** relevant information for later in the string

## Evidence of Syntactic Structure

There is evidence that sequential neural models can capture long-distance dependencies and other aspects of hierarchical syntactic structure.

### Example: Subject-Verb Agreement

```
The keys the man that I saw lost { is  }  ← LSTM will correctly assign
                                  { are }  ← "are" a much higher probability
```

"Probes" like this, and other methods, suggest that deep neural models of language are **implicitly representing structure**

# 17. BERT: Bidirectional Encoder Representations from Transformers

## Overview

- **Word-piece tokens**
- **Masked Language Models**
  - "The man went to the [MASK] to buy a [MASK] of milk."
  - Predicts from context both left-to-right, and right-to-left (hence, "bidirectional")

## Learning Objectives

1. Predict masked word
2. Predict next sentence

## BERT Characteristics

- BERT representations are **powerful** and easily amenable to many fine-tuning tasks
- But they are **complex and expensive to train**

### Multilingual BERT (m-BERT, Devlin et al. 2019)

- Same architecture as BERT, training on a multilingual dataset
- Uses a shared vocabulary of ~100 languages (single model)
- Trained on Wikipedia pages across many different languages

In [ ]:
# Demo: Using BERT for sentence embeddings (requires transformers library)
# Uncomment to install: !pip install transformers

try:
    from transformers import BertTokenizer, BertModel
    import torch
    
    # Load pre-trained BERT
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')
    
    # Example sentences
    sentences = [
        "The bank approved my loan application.",
        "We sat by the river bank."
    ]
    
    print("BERT Sentence Embeddings:")
    for sent in sentences:
        # Tokenize and get embeddings
        inputs = tokenizer(sent, return_tensors="pt")
        outputs = model(**inputs)
        
        # Use [CLS] token embedding as sentence representation
        cls_embedding = outputs.last_hidden_state[0, 0, :]
        print(f"\nSentence: {sent}")
        print(f"Embedding shape: {cls_embedding.shape}")
        print(f"First 5 dimensions: {cls_embedding[:5].detach().numpy()}")
        
except ImportError:
    print("Install transformers library: pip install transformers")
    print("This demo requires the transformers library from HuggingFace")

## 📊 Summary and Key Takeaways

### **Meaning Representation Approaches**

1. **Procedural Semantics**
   - Meaning as procedures (how to make/verify something true)
   - Example: SHRDLU microworld system

2. **Model-theoretic Semantics**
   - Meaning as truth conditions in a model
   - Mapping between representations and world models

3. **Semantic Roles**
   - Predicate-argument structure
   - PropBank, FrameNet
   - Abstract Meaning Representation (AMR)

4. **Compositional Semantics**
   - Lambda calculus for meaning composition
   - Building sentence meaning from parts

### **Neural Approaches to Sentence Meaning**

1. **Recurrent Neural Networks (RNNs)**
   - Handle variable-length sequences
   - Capture sequential dependencies
   - Bidirectional models for full context

2. **Long Short-Term Memory (LSTMs)**
   - Solve vanishing gradient problem
   - Better at long-distance dependencies
   - Gates for controlling information flow

3. **BERT and Transformers**
   - Bidirectional context modeling
   - Masked language modeling
   - State-of-the-art for many tasks

### **Key Applications**

- Semantic Role Labeling (SRL)
- Machine Translation
- Question Answering
- Text Generation
- Sentence Classification

## 🔍 Exercise: Semantic Analysis Pipeline

Build a simple semantic analysis pipeline that:
1. Identifies predicates and arguments
2. Assigns semantic roles
3. Generates a meaning representation

In [ ]:
# Exercise: Complete the semantic analysis pipeline

def semantic_analysis_pipeline(sentence):
    """
    Analyze a sentence and extract semantic information
    
    TODO:
    1. Parse the sentence with spaCy
    2. Identify the main predicate (verb)
    3. Find arguments (subject, object)
    4. Identify modifiers (temporal, locative)
    5. Create a simple meaning representation
    """
    doc = nlp(sentence)
    
    # TODO: Implement the analysis
    predicate = None
    arg0 = None  # subject/agent
    arg1 = None  # object/theme
    modifiers = []
    
    # Your code here
    
    return {
        'predicate': predicate,
        'arg0': arg0,
        'arg1': arg1,
        'modifiers': modifiers
    }

# Test your implementation
test_sentences = [
    "John gave Mary the book yesterday.",
    "The company announced new products in December.",
    "She quickly ate the sandwich."
]

for sent in test_sentences:
    result = semantic_analysis_pipeline(sent)
    print(f"\nSentence: {sent}")
    print(f"Analysis: {result}")

## Next Week Preview

**Topic**: Evaluation Metrics for NLP Systems  
**Focus**: How to measure the quality of semantic analysis, translation, and generation systems  

### Prepare for next week:
- Review precision, recall, and F1 score
- Think about how to evaluate meaning representations
- Consider: What makes a "good" semantic analysis?

---

## Discussion Questions

1. **Representation Trade-offs**: What are the advantages and disadvantages of symbolic vs. neural meaning representations?

2. **Compositionality**: How well do neural models capture compositional semantics compared to lambda calculus?

3. **Long-distance Dependencies**: Why are LSTMs better than simple RNNs for capturing syntactic structure?

4. **BERT vs. Traditional**: How does BERT's approach to meaning differ from traditional semantic role labeling?

5. **Future Directions**: What might be the next breakthrough in sentence-level semantics?